# 01 — Data Ingestion

**Stage 1 of the Tier A pipeline** (`DOCS/STRUCTURE.md`) · Boston Housing, Harrison & Rubinfeld (1978)

> **Purpose of this notebook:** acquire the raw file into the analysis environment *without altering it*,
> prove its schema matches the documented data dictionary, and record enough lineage that another
> analyst can verify they are working from the identical bytes.

**Grain:** one row = one census tract in the Boston Standard Metropolitan Statistical Area, 1970.
**Target:** `MEDV` — median owner-occupied home value, in $1000s.

This notebook writes nothing except an immutable copy of the raw data and two audit files. Every
transformation belongs to Stage 2.

In [1]:
from __future__ import annotations

import hashlib
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Relative paths only -- the repo must be portable.
PROJECT = Path.cwd().parent
SRC_FILE = PROJECT / "housing.csv"
RAW_DIR = PROJECT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)

print(f"python  {sys.version.split()[0]}")
print(f"pandas  {pd.__version__}")
print(f"numpy   {np.__version__}")
print(f"source  {SRC_FILE.relative_to(PROJECT)}  exists={SRC_FILE.exists()}")

python  3.14.6
pandas  3.0.3
numpy   2.5.1
source  housing.csv  exists=True


## 1.1 The file has no header — column names are supplied from the UCI schema

The raw file is whitespace-delimited with **no header row**, so `pandas` would otherwise promote the
first tract to column names and silently lose an observation. Names come from the documented UCI
schema (`DOCS/boston_housing_data_analysis.md` §1).

In [2]:
# UCI schema, in file order. Supplying these is mandatory -- the file carries no header.
COLUMNS = [
    "CRIM",     # per capita crime rate by town
    "ZN",       # proportion of residential land zoned for lots over 25,000 sq.ft.
    "INDUS",    # proportion of non-retail business acres per town
    "CHAS",     # Charles River dummy (1 = tract bounds river)
    "NOX",      # nitric oxides concentration (parts per 10 million)
    "RM",       # average number of rooms per dwelling
    "AGE",      # proportion of owner-occupied units built prior to 1940
    "DIS",      # weighted distance to five Boston employment centres
    "RAD",      # index of accessibility to radial highways
    "TAX",      # full-value property-tax rate per $10,000
    "PTRATIO",  # pupil-teacher ratio by town
    "B",        # 1000(Bk - 0.63)^2, Bk = proportion of Black residents  [RACE-DERIVED]
    "LSTAT",    # % lower status of the population
    "MEDV",     # median value of owner-occupied homes, $1000s  [TARGET]
]

# sep=r"\s+" (not the removed delim_whitespace=) -- pandas 3.x dropped that alias.
raw = pd.read_csv(SRC_FILE, sep=r"\s+", header=None, names=COLUMNS)

print(f"shape: {raw.shape[0]} rows x {raw.shape[1]} columns")
raw.head()

shape: 506 rows x 14 columns


,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2


## 1.2 Lineage — hash, timestamp, and shape are recorded so the source is provable

If the upstream file is ever swapped, the SHA-256 changes and every downstream result becomes
suspect by design.

In [3]:
def sha256_of(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

lineage = pd.DataFrame([
    {"field": "source_file",     "value": SRC_FILE.name},
    {"field": "source_identity", "value": "Boston Housing (Harrison & Rubinfeld 1978), via UCI ML Repository"},
    {"field": "reference_year",  "value": "1970 (Boston SMSA census)"},
    {"field": "sha256",          "value": sha256_of(SRC_FILE)},
    {"field": "bytes",           "value": SRC_FILE.stat().st_size},
    {"field": "ingested_utc",    "value": datetime.now(timezone.utc).isoformat(timespec="seconds")},
    {"field": "n_rows",          "value": raw.shape[0]},
    {"field": "n_columns",       "value": raw.shape[1]},
    {"field": "grain",           "value": "one row = one census tract"},
    {"field": "target",          "value": "MEDV (median home value, $1000s)"},
    {"field": "pandas_version",  "value": pd.__version__},
])
lineage.to_csv(RAW_DIR / "_lineage.csv", index=False)
lineage

,field,value
0,source_file,housing.csv
1,source_identity,"Boston Housing (Harrison & Rubinfeld 1978), vi..."
2,reference_year,1970 (Boston SMSA census)
3,sha256,baadf72995725d76efe787b664e1f083388c79ba21ef9a...
4,bytes,49082
5,ingested_utc,2026-07-20T10:44:04+00:00
6,n_rows,506
7,n_columns,14
8,grain,one row = one census tract
9,target,"MEDV (median home value, $1000s)"


## 1.3 Schema validation — every column is range-checked against the data dictionary

Expectations are declared **before** looking at the data, so a match is evidence rather than a
description of whatever happened to load.

In [4]:
# (column, dtype, min_allowed, max_allowed, rationale) -- declared from the data dictionary.
EXPECTATIONS = [
    ("CRIM",    "float", 0.0,    100.0,  "rate per capita, non-negative"),
    ("ZN",      "float", 0.0,    100.0,  "a proportion expressed 0-100"),
    ("INDUS",   "float", 0.0,    100.0,  "a proportion expressed 0-100"),
    ("CHAS",    "int",   0.0,      1.0,  "binary dummy"),
    ("NOX",     "float", 0.30,     0.90, "parts per 10 million, documented range"),
    ("RM",      "float", 1.0,     15.0,  "rooms per dwelling, physically plausible"),
    ("AGE",     "float", 0.0,    100.0,  "a proportion expressed 0-100"),
    ("DIS",     "float", 0.0,     15.0,  "weighted distance, positive"),
    ("RAD",     "int",   1.0,     24.0,  "accessibility index 1-24"),
    ("TAX",     "float", 100.0,  800.0,  "rate per $10,000"),
    ("PTRATIO", "float", 5.0,     30.0,  "pupils per teacher, plausible"),
    ("B",       "float", 0.0,    400.0,  "1000(Bk-0.63)^2 is bounded above by ~396.9"),
    ("LSTAT",   "float", 0.0,    100.0,  "a percentage"),
    ("MEDV",    "float", 0.0,     60.0,  "$1000s; note top-coding at 50.0"),
]

rows = []
for col, kind, lo, hi, why in EXPECTATIONS:
    s = raw[col]
    is_int_like = float(np.nanmax(np.abs(s - s.round()))) == 0.0
    dtype_ok = is_int_like if kind == "int" else True
    n_below, n_above = int((s < lo).sum()), int((s > hi).sum())
    rows.append({
        "column": col,
        "dtype_expected": kind,
        "dtype_ok": dtype_ok,
        "observed_min": round(float(s.min()), 4),
        "observed_max": round(float(s.max()), 4),
        "allowed_min": lo,
        "allowed_max": hi,
        "n_out_of_range": n_below + n_above,
        "n_null": int(s.isna().sum()),
        "status": "PASS" if (dtype_ok and n_below == 0 and n_above == 0 and s.isna().sum() == 0) else "FAIL",
        "rationale": why,
    })

schema = pd.DataFrame(rows)
schema.to_csv(RAW_DIR / "_schema_validation.csv", index=False)

n_fail = int((schema["status"] == "FAIL").sum())
print(f"schema checks: {len(schema) - n_fail}/{len(schema)} PASS, {n_fail} FAIL")
schema[["column", "observed_min", "observed_max", "allowed_min", "allowed_max",
        "n_out_of_range", "n_null", "status"]]

schema checks: 14/14 PASS, 0 FAIL


,column,observed_min,observed_max,allowed_min,allowed_max,n_out_of_range,n_null,status
0,CRIM,0.0063,88.9762,0.0,100.0,0,0,PASS
1,ZN,0.0000,100.0000,0.0,100.0,0,0,PASS
2,INDUS,0.4600,27.7400,0.0,100.0,0,0,PASS
3,CHAS,0.0000,1.0000,0.0,1.0,0,0,PASS
4,NOX,0.3850,0.8710,0.3,0.9,0,0,PASS
5,RM,3.5610,8.7800,1.0,15.0,0,0,PASS
6,AGE,2.9000,100.0000,0.0,100.0,0,0,PASS
7,DIS,1.1296,12.1265,0.0,15.0,0,0,PASS
8,RAD,1.0000,24.0000,1.0,24.0,0,0,PASS
9,TAX,187.0000,711.0000,100.0,800.0,0,0,PASS


## 1.4 Structural facts confirmed independently, not taken on faith

`DOCS/boston_housing_data_analysis.md` asserts 0 nulls, 0 duplicates, and 16 top-coded records.
Those claims are re-derived here. If the notes and the data disagree, **the data wins**.

In [5]:
claims = pd.DataFrame([
    {"claim": "506 rows x 14 columns",       "documented": "506 x 14",
     "observed": f"{raw.shape[0]} x {raw.shape[1]}"},
    {"claim": "zero missing values",          "documented": "0",
     "observed": str(int(raw.isna().sum().sum()))},
    {"claim": "zero duplicate rows",          "documented": "0",
     "observed": str(int(raw.duplicated().sum()))},
    {"claim": "MEDV top-coded at 50.0",       "documented": "16",
     "observed": str(int((raw["MEDV"] == 50.0).sum()))},
    {"claim": "CHAS imbalance (0 vs 1)",      "documented": "471 vs 35",
     "observed": f"{int((raw.CHAS == 0).sum())} vs {int((raw.CHAS == 1).sum())}"},
    {"claim": "all columns numeric",          "documented": "14",
     "observed": str(int(raw.select_dtypes("number").shape[1]))},
])
claims["agrees"] = claims["documented"] == claims["observed"]
print("all documented claims reproduce:", bool(claims["agrees"].all()))
claims

all documented claims reproduce: True


,claim,documented,observed,agrees
0,506 rows x 14 columns,506 x 14,506 x 14,True
1,zero missing values,0,0,True
2,zero duplicate rows,0,0,True
3,MEDV top-coded at 50.0,16,16,True
4,CHAS imbalance (0 vs 1),471 vs 35,471 vs 35,True
5,all columns numeric,14,14,True


**Every documented claim reproduces.** The data notes are trustworthy, which means the Stage 2
cleaning notebook can focus on *structural decisions* (censoring, dtype semantics) rather than
repair work.

## 1.5 Governance — no PII, but one race-derived column needs classifying

Tract-level aggregates carry no personally identifiable information. One column is nonetheless
sensitive and is classified here so the decision travels with the data rather than being
rediscovered at modelling time.

In [6]:
governance = pd.DataFrame([
    {"column": "B",
     "classification": "SENSITIVE - race-derived aggregate",
     "detail": "1000(Bk-0.63)^2 where Bk is the proportion of Black residents per town.",
     "handling": "EXCLUDED from all models. Retained in raw/processed data for the mandatory "
                 "fairness audit only (proxy test + error-by-quartile). See 04_analysis."},
    {"column": "LSTAT",
     "classification": "SENSITIVE - socioeconomic status",
     "detail": "'% lower status of the population' - a value-laden 1970s construct.",
     "handling": "Retained as a model input; language neutralised in reporting; audited as a "
                 "potential proxy for B."},
    {"column": "all others",
     "classification": "NON-SENSITIVE - tract-level physical/fiscal attributes",
     "detail": "No individual is identifiable from a census-tract aggregate.",
     "handling": "No restriction."},
])
governance.to_csv(RAW_DIR / "_governance.csv", index=False)
for _, r in governance.iterrows():
    print(f"[{r.classification}]\n  {r.column}: {r.detail}\n  -> {r.handling}\n")

[SENSITIVE - race-derived aggregate]
  B: 1000(Bk-0.63)^2 where Bk is the proportion of Black residents per town.
  -> EXCLUDED from all models. Retained in raw/processed data for the mandatory fairness audit only (proxy test + error-by-quartile). See 04_analysis.

[SENSITIVE - socioeconomic status]
  LSTAT: '% lower status of the population' - a value-laden 1970s construct.
  -> Retained as a model input; language neutralised in reporting; audited as a potential proxy for B.

[NON-SENSITIVE - tract-level physical/fiscal attributes]
  all others: No individual is identifiable from a census-tract aggregate.
  -> No restriction.



> **So What:** the `B` column cannot simply be dropped and forgotten. Because `LSTAT` and `CRIM`
> may encode the same information, excluding `B` does **not** by itself make the analysis
> race-neutral — that has to be demonstrated, which is why a proxy test is scheduled for Stage 6.

## 1.6 Write the immutable raw copy

In [7]:
raw_copy = RAW_DIR / "housing.csv"
raw.to_csv(raw_copy, index=False)  # now WITH a header, so downstream reads need no schema guess

verify = pd.read_csv(raw_copy)
assert verify.shape == raw.shape, "round-trip changed the shape"
assert np.allclose(verify.to_numpy(), raw.to_numpy()), "round-trip changed the values"

print(f"wrote {raw_copy.relative_to(PROJECT)}  ({verify.shape[0]} x {verify.shape[1]})")
print("round-trip verified: values identical")
print("\nStage 1 artifacts:")
for f in sorted(RAW_DIR.iterdir()):
    print(f"  data/raw/{f.name:28s} {f.stat().st_size:>8,} bytes")

wrote data\raw\housing.csv  (506 x 14)
round-trip verified: values identical

Stage 1 artifacts:
  data/raw/_governance.csv                   645 bytes
  data/raw/_lineage.csv                      416 bytes
  data/raw/_schema_validation.csv          1,171 bytes
  data/raw/housing.csv                    37,652 bytes


---

## Stage 1 — Gate Checklist

- [x] **Raw data saved to `data/raw/`** — untouched values, round-trip verified against the source
- [x] **Source, timestamp, and row/column counts logged** — `_lineage.csv`, including a SHA-256 so the exact bytes are provable
- [x] **Schema matches expectations** — 14/14 columns pass declared dtype and range checks (`_schema_validation.csv`)
- [x] **PII / data governance review completed** — no PII at tract grain; `B` classified as race-derived and excluded from modelling, `LSTAT` flagged for proxy testing (`_governance.csv`)

**Gate status: PASS → proceed to `02_cleaning.ipynb`.**

### Changelog
| Pass | Date | Change |
|---|---|---|
| 1 | 2026-07-20 | Initial ingestion. All six documented data claims reproduced exactly; no discrepancies to escalate. |